In [ ]:
%load_ext autoreload
%autoreload 2

# Nash-DQN and SRE-DQN Training

Train the Nash and SRE models and save checkpoints into timestamped `pt_files/` folders.

In [ ]:
import os
import random
import numpy as np
import torch
import time
from datetime import datetime

from NashRL import run_Nash_Agent
from NashAgent_lib import NashNN
from SREAgent_lib import SreNN
from SREDQN import run_SRE_Agent
from simulation_lib import MarketSimulator

np.set_printoptions(precision=4)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        torch.use_deterministic_algorithms(True)
    return seed


print('Using device:', 'cuda' if torch.cuda.is_available() else 'cpu')


## Shared Market Environment

Identical parameters to `NashDQN-Training.ipynb`. Both agents are trained on this same environment.

In [ ]:
num_players = 5
kappa = 0.5

sim_dict = {
    'perm_price_impact':  torch.tensor(0.05).cuda().detach(),
    'transaction_cost':   torch.tensor(0.1).cuda().detach(),
    'liquidation_cost':   torch.tensor(0.1).cuda().detach(),
    'running_penalty':    torch.tensor(0.0).cuda().detach(),
    'trans_impact_scale': torch.tensor(0.02).cuda().detach(),
    'trans_impact_decay': torch.tensor(0.5).cuda().detach(),
    'T':                  torch.tensor(5).cuda().detach(),
    'dt':                 torch.tensor(0.5).cuda().detach(),
    'N_agents':           num_players,
    'drift_function':     (lambda x, y: kappa * (10 - y)),
    'volatility':         torch.tensor(0.1).cuda().detach(),
    'init_inv_var':       torch.tensor(50).cuda().detach(),
}

inv_std = sim_dict['volatility'] * torch.sqrt(
    (1 - torch.exp(-2 * kappa * sim_dict['T'])) / (2 * kappa)
)
sim_dict['initial_price_var'] = torch.tensor(inv_std).cuda().detach()

norm_mean = torch.tensor([2.25, 10, 0, 0, 0]).cuda().detach()
norm_std  = torch.tensor([
    1.4361406616345072,
    0.74204157112471332 * 0.2763,
    2.5 * 1.8078,
    0.1 * 0.4225,
    1.0 * 1.6726,
]).cuda().detach()

sim_obj = MarketSimulator(sim_dict, impact='sqrt')
T = sim_dict['T']

print('Agents:', num_players, '| T=', sim_dict['T'].item(), '| dt=', sim_dict['dt'].item())

## Training Configuration


In [ ]:
# Training config
BASE_SEED = 20260409
NASH_SEED = BASE_SEED
SRE_TRAIN_SEED = BASE_SEED

NUM_SIM   = 20000
MAX_STEPS = 10
RV_MIN    = 0.5
RV_MAX    = 2.5
EARLY_STOP = True
EARLY_LIM  = 2000

SRE_EPS_LIST = [0.0, 0.01, 0.1, 0.5, 1.0]
PRIMARY_SRE_EPS = 0.5

RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M')


def eps_slug(eps):
    return f'{eps:g}'.replace('.', 'p')


NASH_MODEL_DIR = os.path.join('pt_files', f'nash_{RUN_TAG}')
SRE_MODEL_DIRS = {
    eps: os.path.join('pt_files', f'sre_eps_{eps_slug(eps)}_{RUN_TAG}')
    for eps in SRE_EPS_LIST
}

# Shared network config (valid for both NashNN and SreNN)
NET_KWARGS = dict(
    non_invar_dim=5,
    output_dim=5,
    n_players=num_players,
    max_steps=MAX_STEPS,
    lr=3e-4,
    weighted_adam=True,
    terminal_cost=sim_dict['liquidation_cost'],
    num_moms=0,
    c_cons=50,
    c2_cons=False,
    c3_pos=False,
    layers=4,
)

# SRE-specific hyperparameters
SRE_EPS_DECAY = None
SRE_EPS_REG = 0.01
SRE_DELTA_MIN = 1e-6
SRE_GAMMA = 1.0

seed_everything(BASE_SEED)

print('Nash model dir:', NASH_MODEL_DIR)
print('Seeds | base:', BASE_SEED, '| nash:', NASH_SEED, '| sre train:', SRE_TRAIN_SEED)
print('NUM_SIM:', NUM_SIM, '| MAX_STEPS:', MAX_STEPS)
print('SRE eps sweep:', SRE_EPS_LIST, '| decay horizon:', SRE_EPS_DECAY)


## Train Nash-DQN

In [ ]:

os.makedirs(NASH_MODEL_DIR, exist_ok=True)

seed_everything(NASH_SEED)
nash_agent = NashNN(**NET_KWARGS)

start = time.time()
nash_agent, nash_loss = run_Nash_Agent(
    sim_obj, sim_dict, MAX_STEPS,
    nash_agent=nash_agent,
    num_sim=NUM_SIM,
    norm_mean=norm_mean,
    norm_std=norm_std,
    rv_min=RV_MIN,
    rv_max=RV_MAX,
    early_stop=EARLY_STOP,
    early_lim=EARLY_LIM,
    path=os.path.join(NASH_MODEL_DIR, ''),
    AN_file_name=os.path.join(NASH_MODEL_DIR, 'Action_Net'),
    VN_file_name=os.path.join(NASH_MODEL_DIR, 'Value_Net'),
)
nash_train_time = time.time() - start
print(f'Nash-DQN training time: {nash_train_time:.1f}s')


## Train SRE-DQN

In [ ]:

sre_agents = {}
sre_losses = {}
sre_train_times = {}

for eps in SRE_EPS_LIST:
    eps_dir = SRE_MODEL_DIRS[eps]
    os.makedirs(eps_dir, exist_ok=True)

    seed_everything(SRE_TRAIN_SEED)
    sre_agent = SreNN(
        **NET_KWARGS,
        eps_reg=SRE_EPS_REG,
        delta_min=SRE_DELTA_MIN,
        gamma=SRE_GAMMA,
    )

    start = time.time()
    trained_agent, train_loss = run_SRE_Agent(
        sim_obj, sim_dict,
        max_steps=MAX_STEPS,
        sre_agent=sre_agent,
        num_sim=NUM_SIM,
        norm_mean=norm_mean,
        norm_std=norm_std,
        rv_min=RV_MIN,
        rv_max=RV_MAX,
        early_stop=EARLY_STOP,
        early_lim=EARLY_LIM,
        eps_0=eps,
        eps_decay_horizon=SRE_EPS_DECAY,
        eps_reg=SRE_EPS_REG,
        delta_min=SRE_DELTA_MIN,
        gamma=SRE_GAMMA,
        path=os.path.join(eps_dir, ''),
        AN_file_name=os.path.join(eps_dir, f'SRE_Action_Net_eps_{eps_slug(eps)}'),
        VN_file_name=os.path.join(eps_dir, f'SRE_Value_Net_eps_{eps_slug(eps)}'),
    )
    sre_agents[eps] = trained_agent
    sre_losses[eps] = train_loss
    sre_train_times[eps] = time.time() - start
    print(f'SRE-DQN training time (eps={eps:g}): {sre_train_times[eps]:.1f}s')
